# H2-1 extension, step 8: placebo test (916 households that never received a campaign)

**Settled so far**: of the 9 type x tier combinations, **only TypeC x high** is still significant after the BH correction
(coef=-6.25, p_adj=0.0033, 295 households, so there is no sample-size problem).

**Still open**: we cannot tell whether this result means "TypeC reduced spending" or "the company sent TypeC at that moment to high-spending
customers whose spending was already falling" (reverse targeting).
Household and week fixed effects filter out "this household's usual level" and "factors common to this week",
but not "why this household received the campaign at exactly this time".

**This step**: using only the 916 households that **never received** a campaign, over the same weeks 33-101,
check whether the high tier shows a pattern of spending falling by itself without any campaign.
If a similar decline also appears in the high tier of non-recipient households, the result just settled is
more likely a natural decline (e.g. regression to the mean) than a campaign effect.

## 0. Environment setup

In [1]:
from importlib.metadata import version
print("linearmodels", version("linearmodels"), "| statsmodels", version("statsmodels"))

linearmodels 7.0 | statsmodels 0.14.6


In [2]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

from config import (DATA_DIR, OUT_DIR, INTERMEDIATE_DIR, FIG_DIR, SEED, TIER_MIN_WEEK, TIER_MAX_WEEK,
                    CAMP_MIN_WEEK, CAMP_MAX_WEEK, MID_WEEK, CELL_SIZE_THRESH, TIER_EN,
                    load_tiers, load_panel, load_camp_full)



## Rebuild the step 1-7 pipeline (same logic, nothing new)

In [3]:
tier_df = load_tiers()
panel = load_panel()[["household_key", "WEEK_NO", "spend", "tier3"]]
recipients = set(tier_df.index[tier_df["will_receive"] == 1])
never_recipients = set(tier_df.index[tier_df["will_receive"] == 0])
print(f"Non-recipient households: {len(never_recipients):,}")
print(f"Full panel shape: {panel.shape}")

Non-recipient households: 916
Full panel shape: (172500, 4)


## 2. Extract only the non-recipient households + check the household counts by tier

As already seen in step 5, the high tier has only 49 non-recipient households, so the sample is small. This has to be kept in mind
when reading the result of this test.

In [4]:
never_panel = panel[panel["household_key"].isin(never_recipients)].copy()

print(f"Non-recipient panel shape: {never_panel.shape}")
print()
print("[Non-recipient households by tier]")
tier_counts_never = never_panel.groupby("tier3", observed=True)["household_key"].nunique()
print(tier_counts_never)
print()
print("Note: the 49 high-tier households are the same sample constraint as in step 5; this test may also have little power")


Non-recipient panel shape: (63204, 4)

[Non-recipient households by tier]
tier3
low     624
mid     243
high     49
Name: household_key, dtype: int64

Note: the 49 high-tier households are the same sample constraint as in step 5; this test may also have little power


## 3. Placebo test A: descriptive comparison (early vs late period mean spend)

The simplest check: without any campaign, does high-tier spending fall in weeks 92-101 compared with weeks 33-42?

In [5]:
early = never_panel[never_panel["WEEK_NO"].between(CAMP_MIN_WEEK, CAMP_MIN_WEEK + 9)]
late  = never_panel[never_panel["WEEK_NO"].between(CAMP_MAX_WEEK - 9, CAMP_MAX_WEEK)]

early_avg = early.groupby("tier3", observed=True)["spend"].mean()
late_avg  = late.groupby("tier3", observed=True)["spend"].mean()

placebo_desc = pd.DataFrame({
    "Early mean (weeks 33-42)": early_avg,
    "Late mean (weeks 92-101)": late_avg,
})
placebo_desc["Change"] = placebo_desc["Late mean (weeks 92-101)"] - placebo_desc["Early mean (weeks 33-42)"]
placebo_desc["Change (%)"] = (placebo_desc["Change"] / placebo_desc["Early mean (weeks 33-42)"] * 100).round(1)
print("[Non-recipient households: spend change from early to late period by tier (descriptive comparison)]")
print(placebo_desc.round(2))


[Non-recipient households: spend change from early to late period by tier (descriptive comparison)]
       Early mean (weeks 33-42)  Late mean (weeks 92-101)  Change  Change (%)
tier3                                                                        
low                        4.61                     12.25    7.63       165.5
mid                       11.27                     17.33    6.06        53.8
high                      43.96                     43.54   -0.42        -1.0


In [6]:
# --- source CSV: descriptive early vs late spending among non-recipients
_d = placebo_desc.rename(columns={"Early mean (weeks 33-42)": "mean_early", "Late mean (weeks 92-101)": "mean_late",
                                  "Change": "change", "Change (%)": "change_pct"}).rename(index=TIER_EN).rename_axis("tier").reset_index()
_d["n_households"] = _d["tier"].map(tier_counts_never.rename(index=TIER_EN))
_d["early_weeks"] = f"{CAMP_MIN_WEEK}-{CAMP_MIN_WEEK + 9}"
_d["late_weeks"] = f"{CAMP_MAX_WEEK - 9}-{CAMP_MAX_WEEK}"
_d.to_csv(OUT_DIR / "placebo_descriptive.csv", index=False)

## 4. Placebo test B: regression-based (household and week fixed effects, same methodology as step 6)

`spend ~ (medium x late period) + (high x late period) + household fixed effects + week fixed effects`, clustered SE.

Compared with the reference (low x late period, and the early period overall), check whether the medium and high tiers show an additional rise or fall
in the late period **even without a campaign**. If this coefficient has a size and direction similar to the step 6 TypeC x high coefficient (-6.25),
it is a warning sign.

In [7]:
never_panel["late"] = (never_panel["WEEK_NO"] >= MID_WEEK).astype(int)
for t in ["mid", "high"]:
    never_panel[f"{t}_x_late"] = ((never_panel["tier3"] == t) & (never_panel["late"] == 1)).astype(int)

placebo_cols = ["mid_x_late", "high_x_late"]

reg_never = never_panel.copy()
reg_never["entity"] = reg_never["household_key"]
reg_never["time"] = reg_never["WEEK_NO"]
reg_never = reg_never.set_index(["entity", "time"])

placebo_model = PanelOLS(
    dependent=reg_never["spend"],
    exog=reg_never[placebo_cols],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
placebo_result = placebo_model.fit(cov_type="clustered", cluster_entity=True)
print(placebo_result.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                  spend   R-squared:                        0.0004
Estimator:                   PanelOLS   R-squared (Between):             -0.0218
No. Observations:               63204   R-squared (Within):              -0.0003
Date:                Mon, Oct 19 2026   R-squared (Overall):             -0.0089
Time:                        15:36:52   Log-likelihood                 -2.98e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      12.216
Entities:                         916   P-value                           0.0000
Avg Obs:                       69.000   Distribution:                 F(2,62218)
Min Obs:                       69.000                                           
Max Obs:                       69.000   F-statistic (robust):             0.6398
                            

In [8]:
# --- source CSV: placebo regression (reference tier = low; week fixed effects absorb the common late-period change)
_pc = placebo_result.conf_int()
pd.DataFrame([{"term": term, "coef": placebo_result.params[term], "se": placebo_result.std_errors[term],
               "ci_low": _pc.loc[term, "lower"], "ci_high": _pc.loc[term, "upper"], "p_value": placebo_result.pvalues[term],
               "late_period_start_week": MID_WEEK, "n_obs": int(placebo_result.nobs),
               "n_households": int(placebo_result.entity_info["total"]),
               "n_households_in_tier": int(tier_counts_never.get("mid" if term.startswith("mid") else "high", 0))}
              for term in placebo_cols]).assign(term=lambda d: d["term"].map({"mid_x_late": "mid_x_late", "high_x_late": "high_x_late"})) \
    .to_csv(OUT_DIR / "placebo_regression.csv", index=False)

## 5. Compare the actual effect and the placebo effect side by side

In [9]:
_fe = pd.read_csv(OUT_DIR / "fe_coefficients.csv")
_c = _fe[(_fe["type"] == "TypeC") & (_fe["tier"] == "high")].iloc[0]
_pcs = pd.read_csv(OUT_DIR / "placebo_regression.csv").set_index("term").loc["high_x_late"]
comparison = pd.DataFrame({
    "comparison": ["[actual] TypeC x high (step 6)", "[placebo] high x late period (non-recipients, step 8)"],
    "coef": [_c["coef"], placebo_result.params.get("high_x_late", np.nan)],
    "p_value": [_c["p_raw"], placebo_result.pvalues.get("high_x_late", np.nan)],
    "n_households": [int(_c["n_recipients"]), int(tier_counts_never.get("high", 0))],
})
pd.DataFrame([
    {"estimate": "actual_typeC_x_high", "coef": _c["coef"], "ci_low": _c["ci_low"], "ci_high": _c["ci_high"], "p_value": _c["p_raw"],
     "n_households": int(_c["n_recipients"])},
    {"estimate": "placebo_high_x_late", "coef": _pcs["coef"], "ci_low": _pcs["ci_low"], "ci_high": _pcs["ci_high"],
     "p_value": _pcs["p_value"], "n_households": int(_pcs["n_households_in_tier"])},
]).to_csv(OUT_DIR / "placebo_vs_actual.csv", index=False)
print(comparison.to_string(index=False))
print()
print("Decision rule:")
print("  placebo coefficient near 0 and not significant  -> the actual effect may still be due to the campaign")
print("  placebo coefficient also negative and significant at a similar size  -> the actual effect may be a natural decline (reverse targeting, etc.)")


                                           comparison      coef  p_value  n_households
                       [actual] TypeC x high (step 6) -6.252591 0.000371           295
[placebo] high x late period (non-recipients, step 8) -4.740102 0.260913            49

Decision rule:
  placebo coefficient near 0 and not significant  -> the actual effect may still be due to the campaign
  placebo coefficient also negative and significant at a similar size  -> the actual effect may be a natural decline (reverse targeting, etc.)


## Interpretation guide

**Order of checking**

1. Look at the sign and significance of the `high_x_late` coefficient in placebo test B.
   - If it is not significant or positive, the TypeC x high result may still be a campaign effect
   - If it is negative and significant, the high tier tends to spend less in the late period even without a campaign,
     so the step 6 result is more likely a natural decline (or endogeneity of the targeting timing itself) than a campaign effect
2. **Always mention the constraint of only 49 households.** Even if the result is "not significant",
   it must be left open to the reading "this sample size may not detect it even if it exists",
   not "strong evidence that there is no placebo effect" (the same limitation already pointed out in step 5).
3. Also check that the descriptive comparison (section 3) and the regression-based one (section 4) point in the same direction. If not, the result is sensitive to the definition of the late period (MID_WEEK)
   or the window, and a further robustness check is needed.

## Wording for the final conclusion

- If the placebo test is passed: "A decline in high-spender spending was observed in TypeC campaign-active weeks, and
  non-recipient households showed no similar pattern over the same period (placebo test passed), so it seems unlikely that this result is
  a natural decline unrelated to the campaign. However, because of the sample constraint (49 households in the high-tier placebo comparison group) and the unknown
  campaign targeting rule, we do not assert a definitive causal relationship."
- If a similar pattern appears in the placebo test: "The TypeC x high effect cannot be distinguished from a natural decline that occurs without a campaign,
  so it is hard to use as a basis for a causal interpretation. Re-verification with a randomized pilot is needed in the future."